# 04. Pipleline Transformacyjny Danych (Batched Chunking dla RAMu)

Wprowadzamy absolutnie pancerną technikę stosowaną przy _Big Data_: **Chunking poprzez izolację urządzeń**.
1. Uczymy metryki matematycznej `StandardScaler Z-Score` na w pełni miarodajnej i wystarczającej losowej próbce stacji (np. rzędu `15%`). Gwarantuje to perfekcyjne złapanie średniej odchylenia całkowicie pomijając ryzyko wybuchu pamięci.
2. Skrypt przepuszcza tabele w malutkich skompresowanych paczkach logiki (po 30 urządzeń na obrót pętli). Przeprowadza inżynierię FE, a następnie dokleja uformowany _DataFrame_ do pliku na dysku `.csv` z bezpiecznym na Windowsie rzutowaniem na Stringi (`encoding=utf-8`).

In [1]:
%load_ext autoreload
%autoreload 2

import polars as pl
from preprocessing import TimeSeriesPreprocessor
import gc
import warnings
import joblib
import numpy as np
import os

warnings.filterwarnings('ignore')

## 1. Wyznaczenie Listy Osi i Kalkulacja Mapy Globalnej (Ochrona Logiczna)

In [2]:
TRAIN_PATH = 'data/data_train.csv'
TEST_PATH = 'data/data_test.csv'

print("1. Skanowanie mapy unikalnych stacji pomiarowych (Device IDs)...")
unique_devices_train = pl.read_csv(TRAIN_PATH, columns=['deviceId']).unique().to_series().to_list()
unique_devices_test = pl.read_csv(TEST_PATH, columns=['deviceId']).unique().to_series().to_list()
print(f"Rozpoznano niezależnych stacji: Train ({len(unique_devices_train)}) | Test ({len(unique_devices_test)})")

print("\n2. Konsolidacja Makro-Sieciowa na pełnej skali tabeli...")
global_stats = pl.scan_csv(TRAIN_PATH).select(['timedate', 'x1']).with_columns(
    pl.col('timedate').str.replace("T", " ").str.slice(0, 19).str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias('dt_obj')
).group_by('dt_obj').agg([
    pl.col('x1').mean().alias('global_net_x1_mean'),
    pl.col('x1').std().alias('global_net_x1_std')
]).collect()

global_stats.write_parquet('data_global_x1_stats.parquet')
print("Pełny obraz topografii sieci sterylnie zapisany -> 'data_global_x1_stats.parquet'")


1. Skanowanie mapy unikalnych stacji pomiarowych (Device IDs)...
Rozpoznano niezależnych stacji: Train (599) | Test (595)

2. Konsolidacja Makro-Sieciowa na pełnej skali tabeli...
Pełny obraz topografii sieci sterylnie zapisany -> 'data_global_x1_stats.parquet'


## 2. Fit Transformatora (Uczenie na próbce OOM-proof)

In [3]:
preprocessor = TimeSeriesPreprocessor(
    weather_path='data/weather.csv', 
    lags=[1, 2, 3, 6, 12, 24],
    global_stats_path='data_global_x1_stats.parquet'
)

print("\n[Faza 1] Adaptacja i uczenie skalera na 20% populacji urządzeń...")
np.random.seed(42)
sample_size = max(1, int(len(unique_devices_train) * 0.20))
sample_devices = np.random.choice(unique_devices_train, size=sample_size, replace=False).tolist()

# Filtrowanie przed załadowaniem do węzła graficznego zapobiega materializowaniu reszty danych
train_lz_sample = pl.scan_csv(TRAIN_PATH).filter(pl.col('deviceId').is_in(sample_devices))
preprocessor.fit(train_lz_sample)

# Zapisz Transformatora pod docelowe użycie na Teście w notatniku 06 (lub inferencji live!)
joblib.dump(preprocessor, 'preprocessor_fitted.joblib')
print("Zapięto klamry metryk. Wytrenowana Instancja bezpieczna w: 'preprocessor_fitted.joblib'")


[Faza 1] Adaptacja i uczenie skalera na 20% populacji urządzeń...
Trwa budowanie matrycy wag i kalkulacja metryk normalizacyjnych głębokich wejść Z-Score. Może to potrwać minuty przy petabajtowych objętościach...
Baza Normalizacyjna ukończona! Posiadam zserializowane parametry skalujące dla 48 cech numerycznych.
Zapięto klamry metryk. Wytrenowana Instancja bezpieczna w: 'preprocessor_fitted.joblib'


## 3. Silnik Przyklejania Transz (Streaming na żywo na Dysk)

In [4]:
def process_and_save_chunks(data_path, device_list, output_name, chunk_size=30):
    print(f"\n[Faza 2] Cięcie pakietowe z ominięciem limitów Pamięci RAM: {output_name}")
    
    if os.path.exists(output_name):
        os.remove(output_name)
        
    total_chunks = int(np.ceil(len(device_list) / chunk_size))
    
    for i in range(0, len(device_list), chunk_size):
        chunk_devs = device_list[i : i + chunk_size]
        print(f"  -> Kalkulacja logiczna węzła {i // chunk_size + 1} z {total_chunks}...")
        
        chunk_lz = pl.scan_csv(data_path).filter(pl.col('deviceId').is_in(chunk_devs))
        transformed_lz = preprocessor.transform(chunk_lz)
        transformed_df = transformed_lz.collect()
        
        write_header = (i == 0)
        if write_header:
            transformed_df.write_csv(output_name, include_header=True)
        else:
            # Zamiana Trybu Binnego zapobiegająca awariom LF -> CRLF w środowisku Windows!
            with open(output_name, 'a', encoding='utf-8', newline='') as f:
                csv_stream = transformed_df.write_csv(include_header=False)
                if isinstance(csv_stream, bytes):
                    csv_stream = csv_stream.decode('utf-8')
                f.write(csv_stream)
                
        del transformed_df
        gc.collect()

    print(f"\n>>> PEŁEN SUKCES LOGICZNY: Przetworzony plik wygenerowany '{output_name}' <<<")

process_and_save_chunks(TRAIN_PATH, unique_devices_train, 'data/transformed_data_train.csv')
process_and_save_chunks(TEST_PATH, unique_devices_test, 'data/transformed_data_test.csv')


[Faza 2] Cięcie pakietowe z ominięciem limitów Pamięci RAM: data/transformed_data_train.csv
  -> Kalkulacja logiczna węzła 1 z 20...
  -> Kalkulacja logiczna węzła 2 z 20...
  -> Kalkulacja logiczna węzła 3 z 20...
  -> Kalkulacja logiczna węzła 4 z 20...
  -> Kalkulacja logiczna węzła 5 z 20...
  -> Kalkulacja logiczna węzła 6 z 20...
  -> Kalkulacja logiczna węzła 7 z 20...
  -> Kalkulacja logiczna węzła 8 z 20...
  -> Kalkulacja logiczna węzła 9 z 20...
  -> Kalkulacja logiczna węzła 10 z 20...
  -> Kalkulacja logiczna węzła 11 z 20...
  -> Kalkulacja logiczna węzła 12 z 20...
  -> Kalkulacja logiczna węzła 13 z 20...
  -> Kalkulacja logiczna węzła 14 z 20...
  -> Kalkulacja logiczna węzła 15 z 20...
  -> Kalkulacja logiczna węzła 16 z 20...
  -> Kalkulacja logiczna węzła 17 z 20...
  -> Kalkulacja logiczna węzła 18 z 20...
  -> Kalkulacja logiczna węzła 19 z 20...
  -> Kalkulacja logiczna węzła 20 z 20...

>>> PEŁEN SUKCES LOGICZNY: Przetworzony plik wygenerowany 'data/transformed